In [193]:
import asyncio
import json
import os
from datetime import datetime
from dotenv import load_dotenv
from typing import List, Literal
from tqdm.asyncio import tqdm_asyncio

from openai import AsyncOpenAI
from pydantic import BaseModel

In [51]:
load_dotenv()

True

In [82]:
def process_single_dataset(dataset_path: str) -> list[dict]:
    with open(dataset_path, "r", encoding="utf-8") as json_file:
        messages = json.load(json_file)["messages"]

    obtained_data = []
    for message in messages:
        if message["type"] == "message" and len(message["text"]) >= 100:
            obtained_data.append({
                "text": message["text"],
                "date": datetime.fromisoformat(message["date"]),
            })

    return obtained_data


In [83]:
def merge_datasets(processed_datasets: list[dict]) -> list[dict]:
    merge_result = []
    for dataset in processed_datasets:
        merge_result.extend(dataset)

    merge_result.sort(key=lambda x: x["date"])

    return merge_result

In [84]:
pozd_proc = process_single_dataset("pozd.json")
but_proc = process_single_dataset("but.json")
merged = merge_datasets([pozd_proc, but_proc])

In [158]:
client = AsyncOpenAI(
    api_key=os.environ.get("OPENROUTER_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

In [183]:
Role = Literal["user", "assistant"]


class DialogMessage(BaseModel):
    role: Role
    content: str


class DatasetSample(BaseModel):
    system_prompt: str
    dialog: List[DialogMessage]


async def process_single_post(post: dict) -> DatasetSample:
    system_prompt = f"""Ты — эксперт по точному воспроизведению стиля Владислава Позднякова.
У тебя есть его реальный пост от {post["date"]}.
Задачи (выполняй строго по порядку):
1. Придумай один реалистичный вопрос от обычного подписчика, на который этот пост звучал бы как естественный и прямой ответ.
    - Вопрос старайся сделать более повседневным, незаурядным и натуральным. Прояви креативность
    - Если пост про какую-то новость, то лучше использовать фразы по типу
    "Как относишься..", "Что думаешь.." и дальше очень кратко про новость (старайся такие вопросы делать покороче)
2. Сформируй ответ ассистента, который должен быть максимально близок к оригинальному посту:
    - сохрани по максимум оригинального текста (80 плюс процентов). Сохрани весь характер, все эмоции и всю экспрессию
    - запрещается убирать маты, нецензурщину, оскорбления и т.д.
    - разрешается усиливать эмоции для предания экспрессии
    - старайся адаптировать ответ под вопрос пользователя, чтобы был полноценный диалог
    - если ответ совсем не имеет смысла, то можешь перефразировать пост, убрать детали, но главное - сохранить характер поста
    - можно убирать текст после слов по типу UPD, если он звучит как бред.
    - можно убирать ссылки на фото и сторонние ресурсы
    - если ответ получается больше 100 слов, урежь его, оставив самое главное и сохранив всю нецензурную лексику!
3. Создай **один короткий и универсальный system prompt** (2–4 предложения), который будет использоваться для всей будущей модели. Он должен максимально точно передавать: уровень и тип мата, любимые конструкции, сарказм, иронию, длину предложений, мировоззрение и все речевые привычки Позднякова.
Никогда не придумывай ничего от себя. Не смягчай мат. Не улучшай стиль.

4. Теперь собери диалог между пользователем и ассистентом на 5–7 сообщений (считая только user/assistant), где:
   - первое сообщение user = вопрос из пункта 1
   - первое сообщение assistant = ответ из пункта 2 (максимально близкий к посту)
   - затем добавь ещё 2–3 пары реплик (user → assistant), чтобы диалог звучал натурально:
       * пользователь уточняет/спорит/просит пример/спрашивает «и что делать?»
       * ассистент отвечает в том же стиле и не добавляет фактов, которых нет в посте (кроме общих рассуждений без конкретики)
   - все ответы ассистента должны сохранять характер, экспрессию и лексику.
Верни результат строго в формате JSON по схеме DatasetSample: system_prompt, dialog (массив сообщений role/content)."""

    response = await client.responses.parse(
        model="x-ai/grok-4.1-fast",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": post["text"]},
        ],
        text_format=DatasetSample,
    )

    sample: DatasetSample = response.output_parsed

    sample.system_prompt = sample.system_prompt + f"\nСегодня {post['date']}"

    return sample

In [204]:
async def __process_all_posts(posts: list[dict]):
    semaphore = asyncio.Semaphore(20)

    async def process_single_post_by_index(post_index: int):
        async with semaphore:
            try:
                result: DatasetSample = await process_single_post(posts[post_index])
                return post_index, result, None
            except Exception as e:
                print(e, flush=True)
                return post_index, None, str(e)

    tasks = [
        process_single_post_by_index(i)
        for i in range(len(posts))
    ]

    results = await tqdm_asyncio.gather(*tasks)

    processed = []
    errored = []
    samples = []

    for post_index, result, error in results:
        if error is None:
            processed.append(post_index)
            samples.append(result.model_dump())
        else:
            errored.append((post_index, error))

    return {
        "processed_posts": processed,
        "posts_with_errors": errored,
        "samples": samples,
    }


async def process_all_posts_and_save(posts: list[dict], output_path: str = "dataset_result.json"):
    result = await __process_all_posts(posts=posts)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


In [205]:
# await process_all_posts(merged)

 17%|█▋        | 1272/7703 [16:20<1:09:07,  1.55it/s]

1 validation error for DatasetSample
  Invalid JSON: EOF while parsing a string at line 1 column 1770 [type=json_invalid, input_value=' {"system_prompt":"Ты ...м шестую.)))}]}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/json_invalid


 28%|██▊       | 2193/7703 [27:41<1:13:43,  1.25it/s]

1 validation error for DatasetSample
  Invalid JSON: EOF while parsing a string at line 1 column 60877 [type=json_invalid, input_value=' {"system_prompt": "Ты...ууууууууууу', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/json_invalid


 29%|██▉       | 2222/7703 [28:00<42:27,  2.15it/s]  

...

 99%|█████████▉| 7661/7703 [31:11<00:00, 205.48it/s]

Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add mo

100%|█████████▉| 7682/7703 [31:11<00:00, 199.08it/s]

Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add mo

100%|██████████| 7703/7703 [31:12<00:00,  4.11it/s] 


In [ ]:
async def __continue_posts_processing(
        last_result_path: str,
        posts: list[dict],
) -> dict[str, list]:
    with open(last_result_path, "r", encoding="utf-8") as json_file:
        last_dataset = json.load(json_file)

    indexes_to_process = last_dataset["posts_with_errors"]
    posts_to_process = [posts[ind] for ind in indexes_to_process]

    result = await __process_all_posts(posts=posts_to_process)

    indexes_to_insert = [indexes_to_process[i] for i in result["processed_posts"]]

    merge_result = []
    all_processed_ids = sorted(set(last_dataset["processed_posts"]) | set(indexes_to_insert))

    current_insert_index = 0
    for old_processed_ind, old_sample in zip(last_dataset["processed_posts"], last_dataset["samples"]):
        while (
            current_insert_index < len(indexes_to_insert)
            and indexes_to_insert[current_insert_index] < old_processed_ind
        ):
            merge_result.append(result["samples"][current_insert_index])
            current_insert_index += 1
        merge_result.append(old_sample)

    while current_insert_index < len(indexes_to_insert):
        merge_result.append(result["samples"][current_insert_index])
        current_insert_index += 1

    return {
        "processed_posts": all_processed_ids,
        "posts_with_errors": result["posts_with_errors"],
        "samples": merge_result,
    }


async def continue_posts_processing_and_save(
        last_result_path: str,
        posts: list[dict],
        output_path: str = "dataset_result.json"
):
    result = await __continue_posts_processing(
        last_result_path=last_result_path,
        posts=posts,
    )

    with open(output_path, "w") as json_file:
        json.dump(result, json_file, ensure_ascii=False, indent=2)
